In [ ]:
import pyreadr
import pandas as pd
import re
import xml
from bs4 import BeautifulSoup
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, classification_report, confusion_matrix
import numpy as np

example_preds = pyreadr.read_r('../../results/example-preds.RData')
example_preds_df = example_preds['pred_df']

#claims_test = pyreadr.read_r('../../data/claims-test.RData')
#claims_test_df = claims_test['claims_test']

claims_clean_ex = pyreadr.read_r('../../data/claims-clean-example.RData')
claims_clean_ex_df = claims_clean_ex['claims_clean']

raw_claims = pyreadr.read_r('../../data/claims-raw.RData')
raw_claims_df = raw_claims['claims_raw']

In [258]:
def parse_fn(html, selector = None):
    soup = BeautifulSoup(html, features = "lxml")
    try:
        for script in soup(["script", "style"]):
            script.decompose() # destroy script and style elements

        if selector is None:
            raw = ' '.join(spoon.get_text(' '))
        else:
            raw = ' '.join(spoon.get_text(' ') for spoon in soup.select(selector))
        
        # remove urls and emails
            # \S@\S+ means non-whitespace characters followed by @ then followed by 1 or more non-whitespace characters
            # pattern = r'https?://\S+|www\.\S+' which means http with optional s followed by :// and then 1 or more non-whitespace characters
            # |www\.\S+ means or www. then 1 or more non-whitespace characters
        text = re.sub(r'\S+@\S+|https?://\S+|www\.\S+', "", raw)

        # remove apostraphes

        text = re.sub("'", "",text)

        # remove non-alphabetic characters and nbsp
        text = re.sub(r'[^a-zA-Z\s]', ' ', text)

        text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text) # add space between camel case words

        text = text.lower()

        text = re.sub(r'\s+', ' ', text).strip() # collapse multiple spaces into one and strip leading/trailing whitespace
        # remove punctuation and newlines
        
        return text 
    except Exception as err:
        print(f"Error parsing HTML: {err}")
        return None


def parse_data(df, selector = None):
    df = df[df['text_tmp'].str.contains('<!', regex = False)].copy() # filter for rows that contain html tags
    df['text_clean'] = df['text_tmp'].apply(parse_fn, selector = selector) # apply parse_fn and create a new column
    return df



In [176]:
clean_claims_baseline = parse_data(raw_claims_df, 'p')
clean_claims_df = parse_data(raw_claims_df, selector = 'h1, h2, h3, h4, h5, h6, p')
# pyreadr.write_rdata("../../data/clean-claims-task1.RData", clean_claims_df, df_name = "dataset")

C:\Users\Felix\AppData\Local\Temp\ipykernel_58236\3546187040.py:2: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, features = "lxml")
C:\Users\Felix\AppData\Local\Temp\ipykernel_58236\3546187040.py:2: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

As

In [259]:

class SVDTxtPipeline:
    
    def __init__(self, prop = 0.7, ngram = None, max_components = 600, random_state = 101422):
        self.lemmatizer = WordNetLemmatizer()
        self.random_state = random_state
        self.prop = prop
        self.max_components = max_components
        # strip punctuation from stop words and lemmatize
        self.stpwrd = {w.replace("'", "") for w in ENGLISH_STOP_WORDS}
        self.tsvd = None
        self.n_pc_ = None
        self.cum_ = None
        # initialize Tfidf object
        if ngram == None:
            self.vectorizer = TfidfVectorizer(tokenizer = self.tokenize, lowercase = False, token_pattern = None)
        elif ngram.lower().strip() == "bigram":
            self.vectorizer = TfidfVectorizer(tokenizer = self.tokenize, ngram_range = (2,2), lowercase = False, token_pattern = None)

    def tokenize(self, text):
        # our tokens, we remove stopwords
        tokens = text.split()
        return [self.lemmatizer.lemmatize(token) for token in tokens if token not in self.stpwrd]
        
    def project(self, train_df):
        # create truncated svd object
        # Fit vectorizer and SVD on training data
        # create dtm with tfidf
        train_dtm = self.vectorizer.fit_transform(train_df['text_clean'])

        # prevent features less than obs
        k = min(self.max_components, train_dtm.shape[1] - 1)
        self.tsvd = TruncatedSVD(n_components = k, random_state = self.random_state)
        self.tsvd.fit(train_dtm)

        # explained variance ratio relative to the total variance
        # get cumsum to explain the prop of variance explained by components
        self.cum_ = self.tsvd.explained_variance_ratio_.cumsum()

        if self.cum_[-1] >= self.prop: # if cum variance exceeds the proportion 
            self.n_pc_ = int(np.argmax(self.cum_ >= self.prop) + 1) # get the first index where cum >= prop and add 1 to compensate for numpy 0-indexings

        else: # if the cum variance never exceeds the proportion
            self.n_pc_ = k # set to k
        

        return self

    def reproject(self, df):
        # transform test data without fit
        dtm = self.vectorizer.transform(df['text_clean'])

        # use transform to multiply the dtm by the first n_pc_ column (from fit)
        # reproject
        return self.tsvd.transform(dtm)[:, :self.n_pc_] 

        
    def process_train_test(self, train_df, test_df):
        # project training data
        self.project(train_df)

        x_train = self.reproject(train_df)
        x_test = self.reproject(test_df)

        y_train = (train_df['bclass'] == "Relevant claim content").astype(int) # convert to binary labels so that relevant = 1
        y_test = (test_df['bclass'] == "Relevant claim content").astype(int)

        return x_train, x_test, y_train, y_test

## Maybe learn to adapt for sklearn pipeline? -_- nah


In [242]:
#### Fit the logistic regression model to the paragraphs and header data set

train_df, test_df = train_test_split(clean_claims_df, test_size = 0.2, random_state = 101422)

claimsf = LogisticRegression(max_iter = 1000)
# Need to learn LogisticRegression documentation :/

pipe = SVDTxtPipeline()
x_train, x_test, y_train, y_test = pipe.process_train_test(train_df, test_df)

claimsf.fit(x_train, y_train)
# predict_proba(x) gives probability estimates. Why is it spelled like that??
probs = claimsf.predict_proba(x_test)[:,1]
preds = (probs > 0.5).astype(int)


In [289]:
## Metrics

print(f"Accuracy: {accuracy_score(y_test, preds)}")
print(f"AUC: {roc_auc_score(y_test, probs)}")
print(f"Sensitivity: {recall_score(y_test, preds)}")
print(f"Specificity: {recall_score(y_test, preds, pos_label = 0)}")
print(f"Confusion matrix: \n {confusion_matrix(y_test, preds)}")
print(f"\n Classification report: \n {classification_report(y_test, preds)}")

Accuracy: 0.8271028037383178
AUC: 0.8934490664919751
Sensitivity: 0.8403755868544601
Specificity: 0.813953488372093
Confusion matrix: 
 [[175  40]
 [ 34 179]]

 Classification report: 
               precision    recall  f1-score   support

           0       0.84      0.81      0.83       215
           1       0.82      0.84      0.83       213

    accuracy                           0.83       428
   macro avg       0.83      0.83      0.83       428
weighted avg       0.83      0.83      0.83       428



In [186]:
#### Fit the logistic regression model to the paragraphs and header data set

train_base_df, test_base_df = train_test_split(clean_claims_baseline, test_size = 0.2, random_state = 101422)

claims_base = LogisticRegression(max_iter = 1000)
# Need to learn LogisticRegression documentation :/

pipe_base = SVDTxtPipeline()
basex_train, basex_test, basey_train, basey_test = pipe_base.process_train_test(train_base_df, test_base_df)

claims_base.fit(basex_train, basey_train)
# predict_proba(x) gives probability estimates. Why is it spelled like that??
base_probs = claims_base.predict_proba(basex_test)[:,1]
base_preds = (base_probs > 0.5).astype(int)


In [290]:
## Metrics

print(f"Accuracy: {accuracy_score(basey_test, base_preds)}")
print(f"AUC: {roc_auc_score(basey_test, base_probs)}")
print(f"Sensitivity: {recall_score(basey_test, base_preds)}")
print(f"Specificity: {recall_score(basey_test, base_preds, pos_label = 0)}")
print(f"Confusion matrix: \n {confusion_matrix(basey_test, base_preds)}")
print(f"\n Classification report: \n {classification_report(basey_test, base_preds)}")

Accuracy: 0.852803738317757
AUC: 0.8834479746697237
Sensitivity: 0.8591549295774648
Specificity: 0.8465116279069768
Confusion matrix: 
 [[182  33]
 [ 30 183]]

 Classification report: 
               precision    recall  f1-score   support

           0       0.86      0.85      0.85       215
           1       0.85      0.86      0.85       213

    accuracy                           0.85       428
   macro avg       0.85      0.85      0.85       428
weighted avg       0.85      0.85      0.85       428



In [170]:
ex_train_df, ex_test_df = train_test_split(claims_clean_ex_df, test_size = 0.2, random_state=101422)
exclaims = LogisticRegression(max_iter = 1000)

ex_pipe = SVDTxtPipeline()
ex_train, ex_test, ey_train, ey_test = ex_pipe.process_train_test(ex_train_df, ex_test_df)

exclaims.fit(ex_train, ey_train)

ex_probs = exclaims.predict_proba(ex_test)[:,1]
ex_preds = (ex_probs > 0.5).astype(int)

In [291]:
## Ex Metrics

print(f"Accuracy: {accuracy_score(ey_test, ex_preds)}")
print(f"AUC: {roc_auc_score(ey_test, ex_probs)}")
print(f"Sensitivity: {recall_score(ey_test, ex_preds)}")
print(f"Specificity: {recall_score(ey_test, ex_preds, pos_label = 0)}")
print(f"Confusion matrix: \n {confusion_matrix(ey_test, ex_preds)}")
print(f"\n Classification report: \n {classification_report(ey_test, ex_preds)}")

Accuracy: 0.8504672897196262
AUC: 0.8840812315754996
Sensitivity: 0.8544600938967136
Specificity: 0.8465116279069768
Confusion matrix: 
 [[182  33]
 [ 31 182]]

 Classification report: 
               precision    recall  f1-score   support

           0       0.85      0.85      0.85       215
           1       0.85      0.85      0.85       213

    accuracy                           0.85       428
   macro avg       0.85      0.85      0.85       428
weighted avg       0.85      0.85      0.85       428



In [ ]:
## Pipe again but with bigrams

biclaims = LogisticRegression(max_iter = 1000)
pipe_bigram = SVDTxtPipeline(ngram = 'bigram')

bix_train, bix_test, biy_train, biy_test = pipe_bigram.process_train_test(train_df, test_df)

biclaims.fit(bix_train, biy_train) # train with only text features
# column 1 is the positive clas probability
bi_probs = biclaims.predict_proba(bix_test)[:,1]

# threshold conversion
bi_preds = (bi_probs > 0.5).astype(int)


NameError: name 'claims_bi' is not defined

In [292]:
## bigram Metrics

print(f"Accuracy: {accuracy_score(biy_test, bi_preds)}")
print(f"AUC: {roc_auc_score(biy_test, bi_probs)}")
print(f"Sensitivity: {recall_score(biy_test, bi_preds)}")
print(f"Specificity: {recall_score(biy_test, bi_preds, pos_label = 0)}")
print(f"Confusion matrix: \n {confusion_matrix(biy_test, bi_preds)}")
print(f"\n Classification report: \n {classification_report(biy_test, bi_preds)}")

Accuracy: 0.8060747663551402
AUC: 0.8816246315099902
Sensitivity: 0.812206572769953
Specificity: 0.8
Confusion matrix: 
 [[172  43]
 [ 40 173]]

 Classification report: 
               precision    recall  f1-score   support

           0       0.81      0.80      0.81       215
           1       0.80      0.81      0.81       213

    accuracy                           0.81       428
   macro avg       0.81      0.81      0.81       428
weighted avg       0.81      0.81      0.81       428



In [ ]:
#  input the predicted log-odds-ratios together with some number of principal components of the bigram-tokenized data to a second logistic regression model.

## We are trying to build a second log regression model to compare the previous model with the text data to decide which are better at 
# predicting the actual outcome

# input means our x scores
# the principal components is in bix_train
# calculate log-odds for train set
bi_probs_train = biclaims.predict_proba(bix_train)[:,1]
log_odds_est_train = np.log(bi_probs_train / (1 - bi_probs_train))

# stack principal components and log odds in matrix
xlog_pc_train = np.column_stack((bix_train, log_odds_est_train))

# Create the test set
log_odds_est_test = np.log(bi_probs / (1 - bi_probs))
xlog_pc_test = np.column_stack((bix_test, log_odds_est_test))

# Train the second log regression
biclaims_2 = LogisticRegression(max_iter = 1000)
# put our combined log-odds-ratios previous pcs with the original labels
# We want to see which is (log-odds-ratio or previous pcs) is more trustworthy for predicting the actual labels
biclaims_2.fit(xlog_pc_train, biy_train) # this is training with text features AND log-odds scores


# get the new probabilities from our new x test set
bi_probs2 = biclaims_2.predict_proba(xlog_pc_test)[:,1]

# threshold conversion
bi_preds2 = (bi_probs2 > 0.5).astype(int)


In [293]:
## bigram meta Metrics
# compare the actual y labels from test set to the predictions from our second "meta" log regression model

print(f"Accuracy: {accuracy_score(biy_test, bi_preds2)}")
print(f"AUC: {roc_auc_score(biy_test, bi_probs2)}")
print(f"Sensitivity: {recall_score(biy_test, bi_preds2)}")
print(f"Specificity: {recall_score(biy_test, bi_preds2, pos_label = 0)}")
print(f"Confusion matrix: \n {confusion_matrix(biy_test, bi_preds2)}")
print(f"\n Classification report: \n {classification_report(biy_test, bi_preds2)}")

Accuracy: 0.8060747663551402
AUC: 0.8863849765258215
Sensitivity: 0.8169014084507042
Specificity: 0.7953488372093023
Confusion matrix: 
 [[171  44]
 [ 39 174]]

 Classification report: 
               precision    recall  f1-score   support

           0       0.81      0.80      0.80       215
           1       0.80      0.82      0.81       213

    accuracy                           0.81       428
   macro avg       0.81      0.81      0.81       428
weighted avg       0.81      0.81      0.81       428

